# Module 4 — Linear Regression: Forecasting Demand in MW
**Lectures:** Week 6 (simple & multiple linear regression, MAE, MSE, RMSE, R²)

Now the real target: **predict demand_mw** for each hour.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ieso_lab import load_dataset
from ieso_lab.features import add_peak_label
from ieso_lab.tracking import log_run, show_runs

plt.rcParams["figure.figsize"] = (9, 4)
df = load_dataset().dropna().reset_index(drop=True)
print(df.shape)

In [ ]:
df = df.sort_values("timestamp_utc").reset_index(drop=True)
cut = int(len(df) * 0.8)
tr, te = df.iloc[:cut], df.iloc[cut:]
y_train, y_test = tr["demand_mw"], te["demand_mw"]

## 1. Metrics by hand
MAE $= \frac1n\sum|y_i-\hat y_i|$ · MSE $= \frac1n\sum(y_i-\hat y_i)^2$ · RMSE $=\sqrt{MSE}$ · $R^2 = 1-\frac{\sum(y_i-\hat y_i)^2}{\sum(y_i-\bar y)^2}$

In [ ]:
def regression_metrics(y, y_hat):
    y, y_hat = np.asarray(y), np.asarray(y_hat)
    # TODO 12: compute mae, rmse, r2 with numpy only
    mae = rmse = r2 = None
    return {"mae": mae, "rmse": rmse, "r2": r2}

print(regression_metrics([200, 300, 400], [250, 300, 350]))  # slides: MAE 33.3, R² 0.75

## 2. Baseline first: "same as yesterday"
Any model must beat the naive forecast $\hat y_t = y_{t-24}$, or it's useless.

In [ ]:
baseline = regression_metrics(y_test, te["demand_lag_24h"])
print("Naive yesterday:", {k: round(v, 3) for k, v in baseline.items()})

## 🔨 3. Break it: a straight line through a U-shape
**Predict:** temperature alone, simple linear regression. Good R² or bad?

In [ ]:
from sklearn.linear_model import LinearRegression
# TODO 13: fit LinearRegression on [["temperature_2m"]] only, predict test
lr1 = None
pred1 = None
m1 = regression_metrics(y_test, pred1)
print(m1, "| slope:", lr1.coef_[0], "MW per °C")

In [ ]:
plt.scatter(te["temperature_2m"], y_test, s=2, alpha=0.3, label="actual")
order = np.argsort(te["temperature_2m"].values)
plt.plot(te["temperature_2m"].values[order], pred1[order], "r", label="fitted line")
plt.xlabel("°C"); plt.ylabel("MW"); plt.legend()

**Why it fails:** demand rises when it's cold (heating) *and* when it's hot (AC). One slope can't do both, so the line averages them into something near-flat.

**Fix with features, not a fancier model:** heating degrees $\text{HDD}=\max(0, 18-T)$ and cooling degrees $\text{CDD}=\max(0, T-18)$. Now the model gets **two slopes** — still linear in its features: $\hat y = b_0 + b_1\text{HDD} + b_2\text{CDD} + \dots$

In [ ]:
FEATURES = ["hdd", "cdd", "hour", "is_weekend", "is_holiday", "demand_lag_24h", "demand_lag_168h"]
# TODO 14: multiple linear regression on FEATURES, predict test, compute metrics
lr = None
pred = None
m = None
print(m)

In [ ]:
print(pd.Series(lr.coef_, index=FEATURES).round(2), "\n(MW change per +1 unit, others fixed)")
log_run("04", "linreg_temp_only", {}, m1)
log_run("04", "linreg_multi", {"features": FEATURES}, m)
log_run("04", "naive_lag24", {}, baseline)

## 🔨 4. Hour isn't linear either
`hour` as one number says demand changes by the same MW every hour, all day. Real demand has a morning ramp and evening peak. Try one-hot encoding hour (`pd.get_dummies(df.hour, prefix="h")`) and compare RMSE.

## Why questions
1. Explain the HDD/CDD coefficients in plain words (MW per degree).
2. Did your model beat the naive baseline? By how much RMSE? If not, why might that be OK?
3. Why does RMSE punish one 2,000 MW miss more than MAE? Which matters more for a grid operator?

---
## 🔧 Git task (do this before you start, and finish after)
```bash
git switch main && git pull
git switch -c module-04-linreg          # start of module
# ... work ...
git add notebooks/ experiments/runs.csv
git commit -m "Module 4: linear regression, HDD/CDD features, baseline"
git push -u origin module-04-linreg
```
Then open a **Pull Request** on GitHub → read your own diff → merge → close the module's Issue.

**Git skill this module:** tag your first real forecaster: `git tag v0.1-linreg && git push --tags`